In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import quad, trapezoid

# =============================================================================
# BAYESIAN COIN: 95% CENTRAL CI + MULTIPLE PRIORS FOR 2D
# Data loaded from actual data.txt (500 flips: 356 heads, 144 tails)
# =============================================================================

# Load actual coin-flip data from data.txt
data = np.loadtxt('data.txt', dtype=int)
n_total = len(data)
n_heads = int(np.sum(data))
n_tails = n_total - n_heads

print("=" * 60)
print("BAYESIAN COIN TOSS: Multiple Priors (Problem 2D)")
print(f"DATA SOURCE: data.txt (actual recorded flips)")
print(f"TOTAL: {n_heads} heads / {n_tails} tails")
print("=" * 60)

def beta_pdf_manual(theta, alpha, beta_param):
    if np.any(theta <= 0) or np.any(theta >= 1):
        return np.zeros_like(theta)
    num = theta**(alpha - 1) * (1 - theta)**(beta_param - 1)
    denom, _ = quad(lambda t: t**(alpha - 1) * (1 - t)**(beta_param - 1), 0, 1)
    return num / denom

def binom_likelihood(theta, n, h):
    return theta**h * (1 - theta)**(n - h)

theta_grid = np.linspace(0.001, 0.999, 500)
n_trials_list = [5, 50, 100, n_total]

# Four different priors (Problem 2D)
priors_list = [
    (2, 5),      # Skeptical of heads (original prior)
    (1, 1),      # Uniform / non-informative
    (10, 10),    # Strong fair-coin belief
    (5, 1)       # Biased toward heads
]
prior_labels = ['Beta(2,5)', 'Beta(1,1)', 'Beta(10,10)', 'Beta(5,1)']

all_stats = []

for prior_idx, (alpha_prior, beta_prior) in enumerate(priors_list):
    print(f"\n--- PRIOR {prior_idx+1}: {prior_labels[prior_idx]} ---")

    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    fig.suptitle(f'Bayesian Coin: Prior Effect (2D)\n'
                 f'{prior_labels[prior_idx]} | Data: {n_heads}H/{n_tails}T',
                 fontsize=14)

    prior_stats = []

    for i, n_trials in enumerate(n_trials_list):
        ax = axes[i // 2, i % 2]

        h = int(np.sum(data[:n_trials])) if n_trials < n_total else n_heads
        t = n_trials - h

        alpha_post = alpha_prior + h
        beta_post = beta_prior + t

        integral_like = trapezoid(binom_likelihood(theta_grid, n_trials, h), theta_grid)
        like_vals = binom_likelihood(theta_grid, n_trials, h) / integral_like

        prior_vals = beta_pdf_manual(theta_grid, alpha_prior, beta_prior)
        post_vals  = beta_pdf_manual(theta_grid, alpha_post, beta_post)

        ax.plot(theta_grid, prior_vals, 'g-', linewidth=2.5, label=f'Prior {prior_labels[prior_idx]}')
        ax.plot(theta_grid, like_vals,  'b-', linewidth=2.5, label='Likelihood (norm.)')
        ax.plot(theta_grid, post_vals,  'r-', linewidth=3.5,
                label=f'Posterior Beta({alpha_post:.0f},{beta_post:.0f})')

        mean_post = alpha_post / (alpha_post + beta_post)
        mode_post = max(0, (alpha_post - 1) / (alpha_post + beta_post - 2))
        mle = h / n_trials

        cum_norm = np.cumsum(post_vals) / np.sum(post_vals)
        ci_low_idx  = np.searchsorted(cum_norm, 0.025)
        ci_high_idx = np.searchsorted(cum_norm, 0.975)
        ci_low  = theta_grid[ci_low_idx]
        ci_high = theta_grid[ci_high_idx]

        ax.axvline(mean_post, color='orange', ls='-', alpha=0.8, linewidth=2,
                   label=f'Mean={mean_post:.3f}')
        ax.axvspan(ci_low, ci_high, alpha=0.25, color='cyan',
                   label=f'95% CI [{ci_low:.3f}, {ci_high:.3f}]')

        ax.set_title(f'{n_trials:,} flips\n({h}H/{t}T | MLE={mle:.3f})')
        ax.set_xlabel(r'Probability of Heads ($\theta$)')
        ax.set_ylabel('Probability Density')
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)

        prior_stats.append([n_trials, h, t, mean_post, mode_post, mle, ci_low, ci_high])

    plt.tight_layout()
    safe_label = prior_labels[prior_idx].replace('(', '').replace(')', '')
    plt.savefig(f'coin_priors_{safe_label}.png', dpi=150, bbox_inches='tight')
    plt.show()

    all_stats.extend([[prior_labels[prior_idx]] + row for row in prior_stats])

# Grand summary
print("\n" + "="*100)
print("PROBLEM 2D: ALL PRIORS SUMMARY")
print("="*100)

df_all = pd.DataFrame(all_stats,
                      columns=['Prior', 'n_trials', 'heads', 'tails',
                                'mean', 'mode', 'MLE', 'CI_low', 'CI_high'])
print(df_all.round(3).to_markdown(index=False))
df_all.to_csv('coin_all_priors_stats.csv', index=False)
print("\n✓ SAVED: coin_priors_[label].png files | coin_all_priors_stats.csv")


BAYESIAN COIN TOSS: Multiple Priors (Problem 2D)
DATA SOURCE: data.txt (actual recorded flips)
TOTAL: 356 heads / 144 tails

--- PRIOR 1: Beta(2,5) ---



--- PRIOR 2: Beta(1,1) ---



--- PRIOR 3: Beta(10,10) ---



--- PRIOR 4: Beta(5,1) ---



PROBLEM 2D: ALL PRIORS SUMMARY
| Prior       |   n_trials |   heads |   tails |   mean |   mode |   MLE |   CI_low |   CI_high |
|:------------|-----------:|--------:|--------:|-------:|-------:|------:|---------:|----------:|
| Beta(2,5)   |          5 |       3 |       2 |  0.417 |  0.4   | 0.6   |    0.167 |     0.693 |
| Beta(2,5)   |         50 |      34 |      16 |  0.632 |  0.636 | 0.68  |    0.503 |     0.751 |
| Beta(2,5)   |        100 |      71 |      29 |  0.682 |  0.686 | 0.71  |    0.591 |     0.767 |
| Beta(2,5)   |        500 |     356 |     144 |  0.706 |  0.707 | 0.712 |    0.665 |     0.745 |
| Beta(1,1)   |          5 |       3 |       2 |  0.571 |  0.6   | 0.6   |    0.223 |     0.881 |
| Beta(1,1)   |         50 |      34 |      16 |  0.673 |  0.68  | 0.68  |    0.541 |     0.793 |
| Beta(1,1)   |        100 |      71 |      29 |  0.706 |  0.71  | 0.71  |    0.615 |     0.789 |
| Beta(1,1)   |        500 |     356 |     144 |  0.711 |  0.712 | 0.712 |    0.671 | 